<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# DXQuant · Derivatives Analytics with Python
## Notebook 9 · Random Numbers and Variance Reduction

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/dxquant/blob/main/notebooks/09_dxquant_random_numbers_and_variance_reduction.ipynb)


## Notebook Goals

DXQuant provides a deterministic standard-normal generator with optional variance reduction:

- antithetic variates, and
- moment matching (sample mean/variance adjustment).

This notebook demonstrates how these options affect a simple European call Monte Carlo
estimator under GBM.

Sampling errors use independent paths or antithetic pair means. A single moment-matched batch reports `NaN` for its error; use independent seeded replications to measure batch dispersion. A shared `scenario_set` explicitly asserts ordered joint scenarios for portfolio payoff risk; it does not construct correlations.


## Setup

In Google Colab, select **Runtime → Run all** with a standard CPU runtime. This cell installs DXQuant and Matplotlib if they are missing. No GPU, Google Drive mount, or API credentials are needed. The examples use synthetic inputs.

Before the first PyPI release, installation uses the public GitHub `main` branch. When running from a local checkout, the setup cell installs the local package instead.


In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys
from pathlib import Path

requirements = []
if importlib.util.find_spec("dxquant") is None:
    package_source = (
        "dxquant @ git+https://github.com/yhilpisch/dxquant.git@main"
    )
    for directory in (Path.cwd(), Path.cwd().parent):
        local_package = directory
        if (local_package / "pyproject.toml").is_file():
            package_source = str(local_package.resolve())
            break
    requirements.append(package_source)
if importlib.util.find_spec("matplotlib") is None:
    requirements.append("matplotlib>=3.8")
if requirements:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *requirements]
    )


In [ ]:
import math

import numpy as np

import dxquant as dx

np.set_printoptions(precision=6, suppress=True)

## 1. Standard-Normal Draws

We compare the sample mean and standard deviation for different settings.

In [ ]:
shape = (3, 10_000)

z_plain = dx.sn_random_numbers(
    shape,
    antithetic=False,
    moment_matching=False,
    seed=123,
)

z_vr = dx.sn_random_numbers(
    shape,
    antithetic=True,
    moment_matching=True,
    seed=123,
)

print("plain mean/std:", float(z_plain.mean()), float(z_plain.std(ddof=1)))
print("vr mean/std:   ", float(z_vr.mean()), float(z_vr.std(ddof=1)))

## 2. European Call by Direct Terminal Simulation

Under GBM, terminal prices have a closed-form representation in terms of a standard normal.
We use that to isolate the variance-reduction effect of the random-number generator.

In [ ]:
spot = 100.0
strike = 105.0
rate = 0.02
vol = 0.20
maturity = 1.0

def call_mc(
    *,
    paths: int,
    antithetic: bool,
    moment_matching: bool,
    seed: int,
) -> tuple[float, float]:
    z = dx.sn_random_numbers(
        (1, paths),
        antithetic=antithetic,
        moment_matching=moment_matching,
        seed=seed,
    )
    z = z.reshape(-1)
    drift = (rate - 0.5 * vol**2) * maturity
    diffusion = vol * math.sqrt(maturity) * z
    terminal = spot * np.exp(drift + diffusion)
    payoff = np.maximum(terminal - strike, 0.0)
    discounted = math.exp(-rate * maturity) * payoff
    price = float(np.mean(discounted))
    if moment_matching or (antithetic and (paths % 2 or paths < 4)):
        std_error = float("nan")  # independent batches needed
    else:
        units = discounted
        if antithetic:
            half = paths // 2
            units = (discounted[:half] + discounted[half:]) / 2
        std_error = float(np.std(units, ddof=1) / math.sqrt(units.size))
    return price, std_error

paths = 80_000

plain = call_mc(
    paths=paths,
    antithetic=False,
    moment_matching=False,
    seed=123,
)

vr = call_mc(
    paths=paths,
    antithetic=True,
    moment_matching=True,
    seed=123,
)

print("plain (price, se):", plain)
print("vr    (price, se):", vr)

## 3. Stability Across Seeds

We compare the dispersion of repeated estimates across different seeds.

In [ ]:
seeds = list(range(20))
paths_small = 20_000

prices_plain = []
prices_vr = []
for seed in seeds:
    p_plain, _ = call_mc(
        paths=paths_small,
        antithetic=False,
        moment_matching=False,
        seed=seed,
    )
    p_vr, _ = call_mc(
        paths=paths_small,
        antithetic=True,
        moment_matching=True,
        seed=seed,
    )
    prices_plain.append(p_plain)
    prices_vr.append(p_vr)

sd_plain = float(np.std(prices_plain, ddof=1))
sd_vr = float(np.std(prices_vr, ddof=1))

print("std dev across seeds (plain):", sd_plain)
print("std dev across seeds (vr):   ", sd_vr)

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">